In [1]:
# Setup
import os
os.makedirs('project_sentinel/models', exist_ok=True)
os.makedirs('project_sentinel/logs', exist_ok=True)
os.makedirs('project_sentinel/profiles', exist_ok=True)
os.makedirs('project_sentinel/data', exist_ok=True)
os.chdir('project_sentinel')
!pip install -q wfdb catboost scikit-learn scipy numpy pandas fastdtw

# config.py
with open('config.py', 'w') as f:
    f.write('''
BASE_SAMPLING_RATE = 128
HIGH_RES_SAMPLING_RATE = 360
BUTTER_ORDER = 4
LOWCUT = 0.5
HIGHCUT = 40.0

RPEAK_MIN_DISTANCE_MS = 200
RPEAK_PROMINENCE = 0.5

RMSSD_WINDOW_SEC = 60
SDNN_WINDOW_SEC = 60
LFHF_WINDOW_SEC = 300

SAMPEN_M = 2
SAMPEN_R_FACTOR = 0.2

NORMAL_SYMBOLS = ['N', 'L', 'R', 'e', 'j']
ARRHYTHMIA_SYMBOLS = ['A', 'a', 'J', 'S', 'V', 'E', 'F', 'P', '/', 'f', 'u']

WINDOW_LENGTH_SEC = 5
CV_FOLDS = 5
CATBOOST_PARAMS = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'F1',
    'random_seed': 42,
    'l2_leaf_reg': 3,
    'verbose': False
}

BPP_EMA_ALPHA = 0.05
BPP_UPDATE_CONSECUTIVE = 8640
BPP_HEALTHY_THRESHOLD = 0.1

CATBOOST_WEIGHT = 0.7
DTW_WEIGHT = 0.3

ALERT_LEVEL_1_LOW = 0.4
ALERT_LEVEL_1_HIGH = 0.6
ALERT_LEVEL_2_LOW = 0.6
ALERT_LEVEL_2_HIGH = 0.8
ALERT_LEVEL_3_LOW = 0.8

STATE_0_TO_1_THRESH = 0.5
STATE_1_TO_2_THRESH = 0.7
STATE_2_STABLE_DURATION = 30

INFERENCE_INTERVAL = {0: 10, 1: 5, 2: 1}
SAMPLING_RATE_BY_STATE = {0: BASE_SAMPLING_RATE, 1: BASE_SAMPLING_RATE, 2: HIGH_RES_SAMPLING_RATE}

MODEL_PATH = 'models/sentinel_model.cbm'
SCALER_PATH = 'models/scaler.pkl'
TRAINING_STATS_PATH = 'models/training_stats.json'
BPP_PATH = 'profiles/bpp_profile.json'
ALERT_LOG_PATH = 'logs/alerts.csv'
''')

# feature_extraction.py
with open('feature_extraction.py', 'w') as f:
    f.write('''
import numpy as np
from scipy.signal import butter, filtfilt, resample_poly, find_peaks, welch
from scipy.interpolate import interp1d
from collections import deque
import config as cfg

def butter_bandpass(lowcut, highcut, fs, order=cfg.BUTTER_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return b, a

def bandpass_filter(signal, fs):
    b, a = butter_bandpass(cfg.LOWCUT, cfg.HIGHCUT, fs)
    return filtfilt(b, a, signal)

def resample_signal(sig, orig_fs, target_fs):
    return resample_poly(sig, target_fs, orig_fs)

def detect_r_peaks(signal, fs):
    dist = int(fs * cfg.RPEAK_MIN_DISTANCE_MS / 1000.0)
    peaks, _ = find_peaks(signal, distance=dist, prominence=cfg.RPEAK_PROMINENCE)
    return peaks

def compute_rmssd(rr_intervals_ms):
    if len(rr_intervals_ms) < 2: return 0.0
    diff = np.diff(rr_intervals_ms)
    return np.sqrt(np.mean(diff ** 2))

def compute_sdnn(rr_intervals_ms):
    if len(rr_intervals_ms) < 2: return 0.0
    return np.std(rr_intervals_ms, ddof=1)

def compute_lf_hf(rr_intervals_ms, fs_resample=4.0):
    if len(rr_intervals_ms) < 10: return 0.0
    rr_sec = np.array(rr_intervals_ms) / 1000.0
    t = np.cumsum(rr_sec) - rr_sec[0]
    t_interp = np.arange(0, t[-1], 1/fs_resample)
    interp_func = interp1d(t, rr_sec, kind='linear', bounds_error=False, fill_value='extrapolate')
    rr_interp = interp_func(t_interp)
    nperseg = min(256, len(rr_interp))
    if nperseg < 32: return 0.0
    freqs, psd = welch(rr_interp, fs=fs_resample, nperseg=nperseg)
    lf_mask = (freqs >= 0.04) & (freqs <= 0.15)
    hf_mask = (freqs >= 0.15) & (freqs <= 0.40)
    lf_power = np.trapz(psd[lf_mask], freqs[lf_mask])
    hf_power = np.trapz(psd[hf_mask], freqs[hf_mask])
    if hf_power == 0: return 0.0 if lf_power == 0 else float('inf')
    return lf_power / hf_power

def compute_sample_entropy(signal, m=cfg.SAMPEN_M, r_factor=cfg.SAMPEN_R_FACTOR):
    N = len(signal)
    if N <= m: return 0.0
    r = r_factor * np.std(signal)
    if r == 0: return 0.0
    def _count_matches(template, data, r):
        dist = np.abs(data - template)
        return np.sum(dist <= r) - 1
    templates_m = np.array([signal[i:i+m] for i in range(N - m)])
    templates_m1 = np.array([signal[i:i+m+1] for i in range(N - m - 1)])
    A = B = 0
    for i in range(len(templates_m)):
        B += _count_matches(templates_m[i], templates_m, r)
    for i in range(len(templates_m1)):
        A += _count_matches(templates_m1[i], templates_m1, r)
    B = B / (N - m) if (N - m) > 0 else 0
    A = A / (N - m - 1) if (N - m - 1) > 0 else 0
    if B == 0: return float('inf')
    return -np.log(A / B)

def compute_baseline_deviation(current_signal_mean, bpp_baseline_mean):
    return abs(current_signal_mean - bpp_baseline_mean)

class FeatureExtractor:
    def __init__(self):
        self.rr_intervals = deque()
    def add_beat(self, timestamp, rr_interval_ms):
        self.rr_intervals.append((timestamp, rr_interval_ms))
        cutoff = timestamp - cfg.LFHF_WINDOW_SEC
        while self.rr_intervals and self.rr_intervals[0][0] < cutoff:
            self.rr_intervals.popleft()
    def get_rr_in_window(self, window_sec, current_time):
        cutoff = current_time - window_sec
        return [rr for t, rr in self.rr_intervals if t >= cutoff and t <= current_time]
    def compute_features(self, current_signal_window, bpp_baseline_mean, current_time):
        rr_60 = self.get_rr_in_window(cfg.RMSSD_WINDOW_SEC, current_time)
        rmssd = compute_rmssd(rr_60)
        sdnn = compute_sdnn(rr_60)
        rr_300 = self.get_rr_in_window(cfg.LFHF_WINDOW_SEC, current_time)
        lf_hf = compute_lf_hf(rr_300)
        sampen = compute_sample_entropy(current_signal_window)
        current_mean = np.mean(current_signal_window)
        baseline_dev = compute_baseline_deviation(current_mean, bpp_baseline_mean)
        return np.array([rmssd, sdnn, lf_hf, sampen, baseline_dev])
''')

# bpp_manager.py
with open('bpp_manager.py', 'w') as f:
    f.write('''
import json
import numpy as np
from datetime import datetime, timezone
import config as cfg

class BPPManager:
    def __init__(self, user_id="sentinel_001"):
        self.user_id = user_id
        self.data = {
            "user_id": user_id,
            "last_updated": datetime.now(timezone.utc).isoformat(),
            "stats": {"rmssd_mean": 0.0, "sdnn_mean": 0.0, "lf_hf_mean": 0.0, "sampen_mean": 0.0},
            "baseline_template": [],
            "normalization_params": {"min": 0.0, "max": 1.0}
        }
    def save(self, path=cfg.BPP_PATH):
        with open(path, 'w') as f: json.dump(self.data, f, indent=2)
    def load(self, path=cfg.BPP_PATH):
        with open(path, 'r') as f: self.data = json.load(f)
        return self.data
    def get_stats(self): return self.data["stats"]
    def get_template(self): return np.array(self.data["baseline_template"])
    def get_baseline_mean(self):
        template = self.get_template()
        return np.mean(template) if len(template) > 0 else 0.0
    def set_template(self, template_ecg): self.data["baseline_template"] = template_ecg.tolist()
    def set_stats(self, rmssd, sdnn, lfhf, sampen):
        self.data["stats"]["rmssd_mean"] = float(rmssd)
        self.data["stats"]["sdnn_mean"] = float(sdnn)
        self.data["stats"]["lf_hf_mean"] = float(lfhf)
        self.data["stats"]["sampen_mean"] = float(sampen)
    def update_from_features(self, features, alpha=cfg.BPP_EMA_ALPHA):
        rmssd, sdnn, lfhf, sampen, _ = features
        s = self.data["stats"]
        s["rmssd_mean"] = alpha * rmssd + (1 - alpha) * s["rmssd_mean"]
        s["sdnn_mean"] = alpha * sdnn + (1 - alpha) * s["sdnn_mean"]
        s["lf_hf_mean"] = alpha * lfhf + (1 - alpha) * s["lf_hf_mean"]
        s["sampen_mean"] = alpha * sampen + (1 - alpha) * s["sampen_mean"]
        self.data["last_updated"] = datetime.now(timezone.utc).isoformat()
''')

# adaptive_engine.py
with open('adaptive_engine.py', 'w') as f:
    f.write('''
import config as cfg

class AdaptiveStateMachine:
    def __init__(self):
        self.state = 0
        self.state2_entry_time = None
        self.stable_seconds_needed = cfg.STATE_2_STABLE_DURATION
    def update(self, anomaly_score, current_time):
        if self.state == 0:
            if anomaly_score > cfg.STATE_0_TO_1_THRESH: self.state = 1
        elif self.state == 1:
            if anomaly_score > cfg.STATE_1_TO_2_THRESH:
                self.state = 2
                self.state2_entry_time = current_time
        elif self.state == 2:
            if anomaly_score < cfg.STATE_0_TO_1_THRESH:
                if self.state2_entry_time is not None and (current_time - self.state2_entry_time) >= self.stable_seconds_needed:
                    self.state = 1
                    self.state2_entry_time = None
            else:
                self.state2_entry_time = current_time
        return self.state
    def get_sampling_rate(self): return cfg.SAMPLING_RATE_BY_STATE[self.state]
    def get_inference_interval(self): return cfg.INFERENCE_INTERVAL[self.state]
''')

# alert_engine.py
with open('alert_engine.py', 'w') as f:
    f.write('''
import config as cfg

_ALERT_MESSAGES = {
    1: "Notice: Your current readings are slightly different from your usual baseline. Monitor how you feel.",
    2: "Alert: We've detected an irregular pattern. Please rest, hydrate, and re-check your status in 15 minutes.",
    3: "Warning: Significant irregular activity detected. Please prioritize your safety and consider consulting a healthcare professional."
}

def evaluate(anomaly_score):
    if anomaly_score < cfg.ALERT_LEVEL_1_LOW: return 0, None
    if anomaly_score < cfg.ALERT_LEVEL_1_HIGH: return 1, _ALERT_MESSAGES[1]
    if anomaly_score < cfg.ALERT_LEVEL_2_HIGH: return 2, _ALERT_MESSAGES[2]
    return 3, _ALERT_MESSAGES[3]
''')

# train.py
with open('train.py', 'w') as f:
    f.write('''
import os, json, pickle, numpy as np
from scipy.signal import resample_poly, butter, filtfilt, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from fastdtw import fastdtw
import wfdb
import config as cfg
from feature_extraction import compute_rmssd, compute_sdnn, compute_lf_hf, compute_sample_entropy, compute_baseline_deviation
from bpp_manager import BPPManager

def butter_bandpass(lowcut, highcut, fs, order=cfg.BUTTER_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return b, a

def bandpass_filter(signal, fs):
    b, a = butter_bandpass(cfg.LOWCUT, cfg.HIGHCUT, fs)
    return filtfilt(b, a, signal)

def resample_signal(sig, orig_fs, target_fs):
    return resample_poly(sig, target_fs, orig_fs)

def detect_r_peaks(signal, fs):
    dist = int(fs * cfg.RPEAK_MIN_DISTANCE_MS / 1000.0)
    peaks, _ = find_peaks(signal, distance=dist, prominence=cfg.RPEAK_PROMINENCE)
    return peaks

def extract_rr(peaks, fs):
    if len(peaks) < 2: return [], []
    times = peaks / fs
    return times[1:], np.diff(times) * 1000.0

def build_bpp(record_name, fs=cfg.BASE_SAMPLING_RATE):
    record = wfdb.rdrecord(record_name, channels=[0])
    sig = resample_signal(record.p_signal[:,0], record.fs, fs)
    sig = bandpass_filter(sig, fs)
    peaks = detect_r_peaks(sig, fs)
    ann = wfdb.rdann(record_name, 'atr')
    normal_mask = np.isin(ann.symbol, cfg.NORMAL_SYMBOLS)
    normal_samples = ann.sample[normal_mask]
    if len(normal_samples) == 0: raise ValueError("No normal annotations")
    mid = len(normal_samples)//2
    ann_time = normal_samples[mid] / record.fs
    idx_128 = int(ann_time * fs)
    half = int(fs * cfg.WINDOW_LENGTH_SEC / 2)
    start = max(0, idx_128 - half)
    end = start + int(fs * cfg.WINDOW_LENGTH_SEC)
    if end > len(sig): end = len(sig); start = end - int(fs * cfg.WINDOW_LENGTH_SEC)
    template = sig[start:end]

    features_list = []
    for samp in normal_samples:
        t = samp / record.fs
        idx = int(t * fs)
        w_start = max(0, idx - half)
        w_end = w_start + int(fs * cfg.WINDOW_LENGTH_SEC)
        if w_end > len(sig): continue
        win_sig = sig[w_start:w_end]
        mask_peaks = peaks / fs <= t
        if np.sum(mask_peaks) < 2: continue
        p_times = peaks[mask_peaks] / fs
        p_rr = np.diff(p_times) * 1000.0
        rr_60 = p_rr[p_times[1:] >= t - 60] if len(p_times)>1 else []
        rr_300 = p_rr[p_times[1:] >= t - 300] if len(p_times)>1 else []
        rmssd = compute_rmssd(rr_60)
        sdnn = compute_sdnn(rr_60)
        lfhf = compute_lf_hf(rr_300)
        sampen = compute_sample_entropy(win_sig)
        features_list.append([rmssd, sdnn, lfhf, sampen, 0.0])
    if features_list:
        farr = np.array(features_list)
        avg_rmssd = np.mean(farr[:,0]); avg_sdnn = np.mean(farr[:,1])
        avg_lfhf = np.mean(farr[:,2]); avg_sampen = np.mean(farr[:,3])
    else:
        avg_rmssd = avg_sdnn = avg_lfhf = avg_sampen = 0.0
    bpp = BPPManager()
    bpp.set_template(template)
    bpp.set_stats(avg_rmssd, avg_sdnn, avg_lfhf, avg_sampen)
    return bpp, sig, peaks, [], []

def process_record(record_name, bpp, fs=cfg.BASE_SAMPLING_RATE):
    record = wfdb.rdrecord(record_name, channels=[0])
    sig = resample_signal(record.p_signal[:,0], record.fs, fs)
    sig = bandpass_filter(sig, fs)
    peaks = detect_r_peaks(sig, fs)
    p_times = peaks / fs
    if len(p_times) < 2: return [], [], []
    p_rr = np.diff(p_times) * 1000.0
    ann = wfdb.rdann(record_name, 'atr')
    half = int(fs * cfg.WINDOW_LENGTH_SEC / 2)
    features, labels, signals = [], [], []
    for sym, samp in zip(ann.symbol, ann.sample):
        if sym in cfg.NORMAL_SYMBOLS: label = 0
        elif sym in cfg.ARRHYTHMIA_SYMBOLS: label = 1
        else: continue
        t = samp / record.fs
        idx = int(t * fs)
        start = max(0, idx - half)
        end = start + int(fs * cfg.WINDOW_LENGTH_SEC)
        if end > len(sig): continue
        win_sig = sig[start:end]
        if len(win_sig) != int(fs * cfg.WINDOW_LENGTH_SEC): continue
        mask = p_times <= t
        if np.sum(mask) < 2: continue
        cur_p_times = p_times[mask]
        cur_rr = np.diff(cur_p_times) * 1000.0
        rr_60 = cur_rr[cur_p_times[1:] >= t - 60] if len(cur_p_times)>1 else []
        rr_300 = cur_rr[cur_p_times[1:] >= t - 300] if len(cur_p_times)>1 else []
        rmssd = compute_rmssd(rr_60)
        sdnn = compute_sdnn(rr_60)
        lfhf = compute_lf_hf(rr_300)
        sampen = compute_sample_entropy(win_sig)
        baseline_dev = compute_baseline_deviation(np.mean(win_sig), bpp.get_baseline_mean())
        features.append([rmssd, sdnn, lfhf, sampen, baseline_dev])
        labels.append(label)
        signals.append(win_sig)
    return features, labels, signals

def compute_dtw_bounds(train_signals, bpp_template):
    template_z = (bpp_template - np.mean(bpp_template)) / (np.std(bpp_template) + 1e-8)
    distances = []
    for sig in train_signals:
        sig_z = (sig - np.mean(sig)) / (np.std(sig) + 1e-8)
        dist, _ = fastdtw(sig_z, template_z)
        distances.append(dist)
    return float(np.min(distances)), float(np.max(distances))

def main():
    print("Downloading MIT-BIH Arrhythmia Database...")
    wfdb.dl_database('mitdb', dl_dir='data')
    records = [f'data/{i}' for i in [100,101,102,103,104,105,106,107,108,109,
                                     111,112,113,114,115,116,117,118,119,
                                     121,122,123,124,200,201,202,203,205,
                                     207,208,209,210,212,213,214,215,217,
                                     219,220,221,222,223,228,230,231,232,233,234]]
    print("Building initial BPP from record 100...")
    bpp, _, _, _, _ = build_bpp('data/100')
    bpp.save()
    all_features, all_labels, all_signals = [], [], []
    for rec in records:
        print(f"Processing {rec}...")
        feats, lbls, sigs = process_record(rec, bpp)
        all_features.extend(feats); all_labels.extend(lbls); all_signals.extend(sigs)
    X = np.array(all_features); y = np.array(all_labels)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print("Training CatBoost...")
    model = CatBoostClassifier(**cfg.CATBOOST_PARAMS)
    skf = StratifiedKFold(n_splits=cfg.CV_FOLDS, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        model.fit(X_scaled[train_idx], y[train_idx], eval_set=(X_scaled[val_idx], y[val_idx]),
                  early_stopping_rounds=50, verbose=False)
        print(f"Fold {fold+1} F1: {model.get_best_score()['validation']['F1']}")
    model.fit(X_scaled, y, verbose=False)
    model.save_model(cfg.MODEL_PATH)
    print("Model saved.")
    with open(cfg.SCALER_PATH, 'wb') as f: pickle.dump(scaler, f)
    print("Computing DTW bounds...")
    dtw_min, dtw_max = compute_dtw_bounds(all_signals, bpp.get_template())
    bpp.data["normalization_params"]["min"] = dtw_min
    bpp.data["normalization_params"]["max"] = dtw_max
    bpp.save()
    train_stats = {"dtw_min": dtw_min, "dtw_max": dtw_max}
    with open(cfg.TRAINING_STATS_PATH, 'w') as f: json.dump(train_stats, f, indent=2)
    print("Training complete. All artefacts saved.")

if __name__ == "__main__":
    main()
''')

# sentinel.py
with open('sentinel.py', 'w') as f:
    f.write('''
import os, csv, json, pickle, numpy as np
from datetime import datetime, timezone
from catboost import CatBoostClassifier
from fastdtw import fastdtw
import wfdb
import config as cfg
from feature_extraction import (FeatureExtractor, bandpass_filter, resample_signal, detect_r_peaks)
from bpp_manager import BPPManager
from adaptive_engine import AdaptiveStateMachine
from alert_engine import evaluate

class SentinelRuntime:
    def __init__(self, test_record='data/100', start_sec=0, duration_sec=60):
        self.test_record = test_record
        self.start_sec = start_sec
        self.duration_sec = duration_sec
        self.model = CatBoostClassifier()
        self.model.load_model(cfg.MODEL_PATH)
        with open(cfg.SCALER_PATH, 'rb') as f: self.scaler = pickle.load(f)
        self.bpp = BPPManager()
        self.bpp.load()
        with open(cfg.TRAINING_STATS_PATH, 'r') as f: self.train_stats = json.load(f)
        self.feat_extractor = FeatureExtractor()
        self.state_machine = AdaptiveStateMachine()
        self._prepare_test_signal()
        os.makedirs(os.path.dirname(cfg.ALERT_LOG_PATH), exist_ok=True)
        self._init_log()

    def _prepare_test_signal(self):
        record = wfdb.rdrecord(self.test_record, channels=[0])
        full_sig = record.p_signal[:, 0]
        sig_128 = resample_signal(full_sig, record.fs, cfg.BASE_SAMPLING_RATE)
        sig_128 = bandpass_filter(sig_128, cfg.BASE_SAMPLING_RATE)
        total_len = len(sig_128)
        start_idx = int(self.start_sec * cfg.BASE_SAMPLING_RATE)
        end_idx = min(start_idx + int(self.duration_sec * cfg.BASE_SAMPLING_RATE), total_len)
        self.stream_signal = sig_128[start_idx:end_idx]
        self.stream_start_time = start_idx / cfg.BASE_SAMPLING_RATE

    def _init_log(self):
        if not os.path.isfile(cfg.ALERT_LOG_PATH):
            with open(cfg.ALERT_LOG_PATH, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['timestamp', 'state', 'anomaly_score', 'alert_level', 'message'])

    def _log_alert(self, state, score, level, message):
        with open(cfg.ALERT_LOG_PATH, 'a', newline='') as f:
            csv.writer(f).writerow([
                datetime.now(timezone.utc).isoformat(), state,
                round(score, 4), level, message if message else ''
            ])

    def _compute_anomaly_score(self, current_signal, current_time):
        features = self.feat_extractor.compute_features(current_signal,
                                                        self.bpp.get_baseline_mean(), current_time)
        X = self.scaler.transform(features.reshape(1, -1))
        proba = self.model.predict_proba(X)[0][1]
        template = self.bpp.get_template()
        sig_z = (current_signal - np.mean(current_signal)) / (np.std(current_signal) + 1e-8)
        tmpl_z = (template - np.mean(template)) / (np.std(template) + 1e-8)
        dtw_dist, _ = fastdtw(sig_z, tmpl_z)
        dtw_min = self.train_stats['dtw_min']
        dtw_max = self.train_stats['dtw_max']
        dtw_norm = (dtw_dist - dtw_min) / (dtw_max - dtw_min) if dtw_max > dtw_min else 0.0
        dtw_norm = max(0.0, min(1.0, dtw_norm))
        return cfg.CATBOOST_WEIGHT * proba + cfg.DTW_WEIGHT * dtw_norm

    def run(self):
        fs = cfg.BASE_SAMPLING_RATE
        sig = self.stream_signal
        peaks = detect_r_peaks(sig, fs)
        peak_times = peaks / fs + self.stream_start_time
        rr = np.diff(peak_times) * 1000.0
        for pt, rri in zip(peak_times[1:], rr):
            self.feat_extractor.add_beat(pt, rri)
        current_time = self.stream_start_time
        end_time = current_time + len(sig)/fs
        next_inference = current_time
        while current_time < end_time:
            state = self.state_machine.state
            interval = self.state_machine.get_inference_interval()
            next_inference += interval
            if next_inference > end_time: break
            win_dur = cfg.WINDOW_LENGTH_SEC
            win_end_idx = int((next_inference - self.stream_start_time) * fs)
            win_start_idx = max(0, win_end_idx - int(win_dur * fs))
            if win_end_idx > len(sig): break
            current_win = sig[win_start_idx:win_end_idx]
            if len(current_win) < int(win_dur * fs): continue
            score = self._compute_anomaly_score(current_win, next_inference)
            new_state = self.state_machine.update(score, next_inference)
            level, msg = evaluate(score)
            if level > 0:
                print(f"[{next_inference:.1f}s] State {new_state} | Score: {score:.3f} | Alert L{level}: {msg}")
                self._log_alert(new_state, score, level, msg)
            else:
                print(f"[{next_inference:.1f}s] State {new_state} | Score: {score:.3f}")
            current_time = next_inference
        print("Simulation complete. Alerts logged to logs/alerts.csv")

if __name__ == "__main__":
    rt = SentinelRuntime(test_record='data/100', start_sec=0, duration_sec=120)
    rt.run()
''')

# Run
print("=== Starting training ===")
!python train.py
print("=== Starting live simulation ===")
!python sentinel.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
=== Starting training ===
Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating r